# Final Forecast & Business Insights

## Objective

Train the final gradient boosting model on all available historical data, generate a forecast for the next 8 weeks per store, break down historical error by store to identify which stores are hardest to forecast, and translate results into concrete business decisions (inventory, staffing, store prioritization).

## Final Model Training

**Decision:** Retrain gradient boosting on the full model_data (all 4,095 rows), not just train_set, since we're no longer holding out data for evaluation — backtesting already confirmed this model beats the baseline across 4 independent windows.

In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

model_data = pd.read_csv("../data/processed/store_week_features.csv", parse_dates=["Date"])

feature_cols = ["lag_1", "lag_2", "lag_52", "rolling_mean_4", "rolling_std_4",
                "WeekOfYear", "Month", "HolidayName", "Type", "Size"]

full_data = model_data.dropna(subset=feature_cols)
full_encoded = pd.get_dummies(full_data[feature_cols + ["Weekly_Sales"]], columns=["HolidayName", "Type"])

X_full = full_encoded.drop(columns=["Weekly_Sales"])
y_full = full_encoded["Weekly_Sales"]

final_model = GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)
final_model.fit(X_full, y_full)

print("Final model trained on", len(X_full), "rows")
print("Date range used:", full_data["Date"].min(), "to", full_data["Date"].max())

Final model trained on 4095 rows
Date range used: 2011-02-04 00:00:00 to 2012-10-26 00:00:00


## Store-Level Historical Error

**Decision:** Use the backtested predictions already computed (from 06_backtesting_and_model_comparison.ipynb's last window) to check per-store error, and compare against store Type.

In [8]:
# Recreate the last backtest window's predictions, but keep store-level detail
cutoff = model_data["Date"].sort_values().unique()[-9]  # 8 weeks before the end
train_final = model_data[model_data["Date"] <= cutoff].dropna(subset=feature_cols)
test_final = model_data[model_data["Date"] > cutoff].dropna(subset=feature_cols)

train_enc = pd.get_dummies(train_final[feature_cols + ["Weekly_Sales", "Store"]], columns=["HolidayName", "Type"])
test_enc = pd.get_dummies(test_final[feature_cols + ["Weekly_Sales", "Store"]], columns=["HolidayName", "Type"])
train_enc, test_enc = train_enc.align(test_enc, join="left", axis=1, fill_value=0)

X_tr = train_enc.drop(columns=["Weekly_Sales", "Store"])
y_tr = train_enc["Weekly_Sales"]
X_te = test_enc.drop(columns=["Weekly_Sales", "Store"])
y_te = test_enc["Weekly_Sales"]

store_model = GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)
store_model.fit(X_tr, y_tr)
test_enc["pred"] = store_model.predict(X_te)
test_enc["abs_error"] = np.abs(test_enc["Weekly_Sales"] - test_enc["pred"])

store_error = test_enc.groupby("Store").apply(
    lambda d: pd.Series({
        "WAPE": d["abs_error"].sum() / d["Weekly_Sales"].abs().sum(),
        "n_weeks": len(d)
    })
)

# Bring back store Type for comparison
store_types = model_data[["Store","Type"]].drop_duplicates().set_index("Store")
store_error = store_error.join(store_types)

print("Worst 10 stores by WAPE:")
print(store_error.sort_values("WAPE", ascending=False).head(10))
print()
print("Average WAPE by Type:")
print(store_error.groupby("Type")["WAPE"].mean())

Worst 10 stores by WAPE:
           WAPE  n_weeks Type
Store                        
14     0.113905      8.0    A
7      0.070357      8.0    B
17     0.060911      8.0    B
18     0.057634      8.0    B
28     0.055580      8.0    A
2      0.051278      8.0    A
35     0.048977      8.0    B
16     0.045460      8.0    B
11     0.044291      8.0    A
21     0.042624      8.0    B

Average WAPE by Type:
Type
A    0.036099
B    0.038998
C    0.030153
Name: WAPE, dtype: float64


/var/folders/l0/_xc3x7m10hxf3hjl2cj6gz2c0000gn/T/ipykernel_10848/2406018594.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  store_error = test_enc.groupby("Store").apply(


### Findings: Store-Level Forecast Error

**Observed:** Average WAPE by Type on this 8-week backtest window: Type A 0.036, Type B 0.039, Type C 0.030. Type B is the worst on average, consistent with the Phase 4 finding that Type B stores had the highest coefficient of variation in historical sales. However, this contradicts the seasonal-naive baseline's Type breakdown from Phase 6 (where Type C was worst at 0.058) — gradient boosting appears to handle Type C's pattern much better than a pure year-over-year lookup does, while Type B remains comparatively hard regardless of model choice.

The single worst-performing store is Store 14 (Type A, WAPE 0.114) — more than double the error of the next-worst store (Store 7, WAPE 0.070). This is a genuine outlier, not representative of Type A as a whole (Type A's average WAPE of 0.036 would be noticeably lower without Store 14 pulling it up).

**Decision:** Flag Store 14 individually for manual review/closer forecast oversight rather than trusting the automated forecast at face value. Type B stores as a group warrant a wider safety margin in inventory planning (e.g., higher safety stock) given their consistently higher error across both baseline and gradient boosting models.

**Why:** A single extreme outlier (Store 14) would be hidden if we only reported Type-level averages — this is exactly why store-level detail matters for real business decisions, not just aggregate model metrics.

## Future Forecast

**Decision:** Generate an 8-week-ahead forecast per store using the final model trained on all available data. Since true future weeks have no actual lag_1/lag_2/rolling features yet, forecast iteratively: predict week 1, use that prediction to build week 2's lag_1, and so on.

In [29]:
last_date = model_data["Date"].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=8, freq="W-FRI")

print("Forecasting for:", future_dates.tolist())

# Build a working copy of the most recent history per store to iteratively forecast forward
history = model_data.sort_values(["Store","Date"]).copy()
forecasts = []

holiday_weeks_future = {"2012-11-23": "Thanksgiving"}
holiday_map_future = {pd.Timestamp(k): v for k, v in holiday_weeks_future.items()}

last_date = model_data["Date"].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=8, freq="W-FRI")

print("Forecasting for:", future_dates.tolist())

history = model_data.sort_values(["Store","Date"]).copy()
forecasts = []

for future_date in future_dates:
    week_of_year = future_date.isocalendar()[1]
    month = future_date.month

    rows_to_predict = []
    for store_id in sorted(history["Store"].unique()):
        store_hist = history[history["Store"] == store_id].sort_values("Date")
        last_4 = store_hist["Weekly_Sales"].tail(4)
        lag_1 = store_hist["Weekly_Sales"].iloc[-1]
        lag_2 = store_hist["Weekly_Sales"].iloc[-2]
        target_lag52_date = future_date - pd.Timedelta(weeks=52)
        lag52_row = store_hist[store_hist["Date"] == target_lag52_date]
        lag_52 = lag52_row["Weekly_Sales"].values[0] if len(lag52_row) > 0 else store_hist["Weekly_Sales"].mean()

        rows_to_predict.append({
            "Store": store_id,
            "Date": future_date,
            "lag_1": lag_1, "lag_2": lag_2, "lag_52": lag_52,
            "rolling_mean_4": last_4.mean(), "rolling_std_4": last_4.std(),
            "WeekOfYear": week_of_year, "Month": month,
            "HolidayName": holiday_map_future.get(future_date, "NoHoliday"),
            "Type": store_hist["Type"].iloc[-1],
            "Size": store_hist["Size"].iloc[-1]
        })

    pred_df = pd.DataFrame(rows_to_predict)
    pred_encoded = pd.get_dummies(pred_df[feature_cols], columns=["HolidayName", "Type"])
    pred_encoded = pred_encoded.reindex(columns=X_full.columns, fill_value=0)
    pred_df["Weekly_Sales"] = final_model.predict(pred_encoded)

    forecasts.append(pred_df[["Store","Date","Weekly_Sales"]])
    history = pd.concat([history, pred_df[["Store","Date","Weekly_Sales","Type","Size"]]], ignore_index=True)

future_forecast = pd.concat(forecasts, ignore_index=True)
print(future_forecast.shape)
print(future_forecast.head(10))

Forecasting for: [Timestamp('2012-11-02 00:00:00'), Timestamp('2012-11-09 00:00:00'), Timestamp('2012-11-16 00:00:00'), Timestamp('2012-11-23 00:00:00'), Timestamp('2012-11-30 00:00:00'), Timestamp('2012-12-07 00:00:00'), Timestamp('2012-12-14 00:00:00'), Timestamp('2012-12-21 00:00:00')]
Forecasting for: [Timestamp('2012-11-02 00:00:00'), Timestamp('2012-11-09 00:00:00'), Timestamp('2012-11-16 00:00:00'), Timestamp('2012-11-23 00:00:00'), Timestamp('2012-11-30 00:00:00'), Timestamp('2012-12-07 00:00:00'), Timestamp('2012-12-14 00:00:00'), Timestamp('2012-12-21 00:00:00')]
(360, 3)
   Store       Date  Weekly_Sales
0      1 2012-11-02  1.760813e+06
1      2 2012-11-02  1.969683e+06
2      3 2012-11-02  4.503712e+05
3      4 2012-11-02  2.295305e+06
4      5 2012-11-02  3.351728e+05
5      6 2012-11-02  1.551508e+06
6      7 2012-11-02  5.663157e+05
7      8 2012-11-02  9.614749e+05
8      9 2012-11-02  6.191271e+05
9     10 2012-11-02  1.891189e+06


In [31]:
check = future_forecast[future_forecast["Store"].isin([1, 4, 20])].pivot(index="Date", columns="Store", values="Weekly_Sales")
print(check)

Store                 1             4             20
Date                                                
2012-11-02  1.760813e+06  2.295305e+06  2.213245e+06
2012-11-09  1.645192e+06  2.261966e+06  2.241561e+06
2012-11-16  1.631092e+06  2.275508e+06  2.177538e+06
2012-11-23  1.992585e+06  3.008179e+06  2.869320e+06
2012-11-30  1.652394e+06  2.160527e+06  2.244035e+06
2012-12-07  1.812683e+06  2.565188e+06  2.482759e+06
2012-12-14  1.911192e+06  2.803756e+06  2.628630e+06
2012-12-21  2.212722e+06  3.593563e+06  3.587876e+06


### Findings: Future Forecast

**Observed:** The model generates an 8-week forecast (2012-11-02 to 2012-12-21) for all 45 stores by iteratively predicting one week at a time and feeding each prediction back in as the next week's lag_1. This period includes Thanksgiving (2012-11-23) and approaches Christmas, so it's a meaningful test of whether the model's learned holiday behavior generalizes to true out-of-sample future dates, not just backtested historical weeks. Predicted sales scale sensibly with store size and Type (e.g., large Type A stores predict in the $1.5M-$2.3M range per week, smaller stores in the low hundreds of thousands), consistent with everything observed in the time-series and store-level analysis.

**Correction:** The initial version of this forecast incorrectly set HolidayName to "NoHoliday" for every future week, missing the Thanksgiving effect for 2012-11-23. Fixed by mapping known future holiday dates before prediction. Verified: Stores 1, 4, and 20 all show a clear spike in predicted sales for 2012-11-23 (roughly 20-32% above surrounding weeks), consistent with the Thanksgiving pattern found in the historical time-series analysis, then return to baseline the following week.